# Transofrmer Architecture Implementation

## Setup and Import

In [1]:
import torch
from torch import nn, optim
from torch.nn import functional as F

import numpy as np

## Positional Encoding

In [2]:
def positional_encoding(seq_length, d, n=10_000):
    P = torch.zeros((seq_length, d)) # Initiate the matrix

    pos = torch.arange(seq_length).unsqueeze(1) # Get all positions
    dim_idx = torch.arange(0, d, 2) # Get all dimensions

    denominator = torch.pow(n, dim_idx/d) # Calculate the denominator (n^(2i/d))
    P[:, 0::2] = torch.sin(pos/denominator) # Get the positional encoding for all even positions
    P[:, 1::2] = torch.cos(pos/denominator) # Get the positional encoding for all odd positions
    return P

In [3]:
input_ids = torch.tensor([[2, 3, 1, 5, 4]]) # 5×1 (batch size * sequence length)
seq_length = input_ids.shape[1]*2

embeddings_dim = 512
embedding = nn.Embedding(embedding_dim=embeddings_dim, num_embeddings=seq_length)
embeddings = embedding(input_ids)


P = positional_encoding(seq_length=input_ids.shape[1], d=512)

final_word_embeddings = embeddings + P
final_word_embeddings # Original Word Embedding + Positional Encoding

tensor([[[ 2.2178,  2.3665, -0.3811,  ...,  0.5654, -0.1945, -1.0519],
         [ 0.6249,  1.6876, -1.0116,  ...,  3.0143, -0.3148,  2.7527],
         [ 1.8838,  1.5655, -0.2175,  ...,  1.5896,  0.0484,  0.5591],
         [ 1.5099, -0.9242,  1.3687,  ...,  0.6374, -0.1759,  2.6006],
         [-0.7758, -1.1027, -0.7186,  ...,  0.7596,  0.1101,  1.0069]]],
       grad_fn=<AddBackward0>)

## Self Attention Head

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.q = nn.Linear(in_features=embedding_dim, out_features=embedding_dim) # Query (W_Q . X)
        self.k = nn.Linear(in_features=embedding_dim, out_features=embedding_dim) # Key (W_K . X)
        self.v = nn.Linear(in_features=embedding_dim, out_features=embedding_dim) # Value (W_V . X)

    def forward(self, x):
        Q = self.q(x) # Train the Query weights and get the Query
        K = self.k(x) # Train the Key weights and get the Key
        V = self.v(x) # Train the Value weights and get the Value

        # Calculate the numerator of the attention equation (the dot-product attention)
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # Get D_K (The dimension size of K and Q)
        d_k = K.size(-1)

        # Calculate the attention weights (Normalize the weights)
        attention_weights = F.softmax(scores/np.sqrt(d_k), dim=-1)

        # Calculate the attention (attention_weights . V)
        attention = torch.matmul(attention_weights, V)
        return attention_weights, attention

In [5]:
vocab_size = 10

attention_head = SelfAttention(embeddings_dim)

attn_weights, single_head_output = attention_head(final_word_embeddings)
attn_weights, single_head_output # Each row in the attention weights represents the attention weights of a word. and each row in the output is the vector representation of a word

(tensor([[[0.1655, 0.0782, 0.2984, 0.1986, 0.2593],
          [0.1701, 0.1510, 0.1780, 0.2498, 0.2511],
          [0.2370, 0.1473, 0.2323, 0.1689, 0.2145],
          [0.1495, 0.1121, 0.2494, 0.1302, 0.3587],
          [0.1985, 0.1564, 0.2600, 0.1849, 0.2002]]],
        grad_fn=<SoftmaxBackward0>),
 tensor([[[-0.0080,  0.8073,  0.9409,  ..., -0.0279,  0.7442,  0.4817],
          [-0.0043,  0.7029,  0.9788,  ..., -0.0018,  0.6717,  0.3782],
          [-0.0442,  0.6554,  0.9521,  ..., -0.0515,  0.7353,  0.4623],
          [-0.0588,  0.8422,  0.9219,  ..., -0.1100,  0.6673,  0.5106],
          [-0.0156,  0.6424,  0.9447,  ..., -0.0274,  0.7410,  0.4344]]],
        grad_fn=<UnsafeViewBackward0>))

In [6]:
single_head_output.shape

torch.Size([1, 5, 512])

## Multi-Head Attention

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, embed_size):
        super().__init__()
        assert embed_size % num_heads == 0, "Embedding size must be divisible by number of heads"
        self.head_dim = embed_size // num_heads # Each head should be of dimension embedding_size/number_of_heads
        self.q = nn.Linear(in_features=embed_size, out_features=embed_size) # Query (W_Q . X)
        self.k = nn.Linear(in_features=embed_size, out_features=embed_size) # Key (W_K . X)
        self.v = nn.Linear(in_features=embed_size, out_features=embed_size) # Value (W_V . X)
        self.num_heads = num_heads # Number of attenton heads

        # Output Fully Connected layer
        self.fc = nn.Linear(in_features=embed_size, out_features=embed_size)

    def forward(self, query, key, value, mask=None):
        B = query.size(0) # Batch Size
        T_query = query.size(1) # Sequence Length for Query
        T_key = key.size(1) # Sequence Length for Key
        Q = self.q(query).reshape(B, T_query, self.num_heads, self.head_dim).transpose(1, 2) # Train the Query weights and get the Query
        K = self.k(key).reshape(B, T_key, self.num_heads, self.head_dim).transpose(1, 2) # Train the Key weights and get the Key
        V = self.v(value).reshape(B, T_key, self.num_heads, self.head_dim).transpose(1, 2) # Train the Value weights and get the Value

        scores = torch.matmul(Q, K.transpose(-2, -1))
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(1), float("-inf"))

        # Get D_K (The dimension size of K and Q)
        d_k = K.size(-1)

        # Calculate the attention weights (Normalize the weights)
        attention_weights = F.softmax(scores/np.sqrt(d_k), dim=-1)

        # Calculate the attention (attention_weights . V)
        attention = torch.matmul(attention_weights, V)
        output = attention.transpose(1, 2).contiguous().reshape(B, T_query, -1)
        return self.fc(output)

In [8]:
embeddings_dim = 512

mha = MultiHeadAttention(num_heads=8, embed_size=embeddings_dim)

mha_output = mha(final_word_embeddings, final_word_embeddings, final_word_embeddings)
mha_output # Each row in the output is the vector representation of a word

tensor([[[ 0.3153,  0.0302, -0.0612,  ...,  0.3111,  0.4469, -0.4520],
         [ 0.2553,  0.0195, -0.0658,  ...,  0.4297,  0.4593, -0.4053],
         [ 0.3316, -0.0727, -0.0211,  ...,  0.3325,  0.5191, -0.4681],
         [ 0.3423, -0.0204, -0.0636,  ...,  0.2842,  0.3979, -0.4497],
         [ 0.2926,  0.0066, -0.0894,  ...,  0.2841,  0.3894, -0.4898]]],
       grad_fn=<ViewBackward0>)

## Encoder-Decorder

### Encoder Block

In [9]:
class EncoderBlock(nn.Module):
    def __init__(self, num_heads, embed_size, dropout=0.1, expansion_factor=4):
        super().__init__()
        # Sub-Layer 1
        self.mha = MultiHeadAttention(num_heads, embed_size)
        self.dropout_1 = nn.Dropout(p=dropout)

        # LayerNorm 1
        self.layer_norm_1 = nn.LayerNorm(embed_size)

        # Sub-Layer 2
        self.ffnn = nn.Sequential(
            nn.Linear(in_features=embed_size, out_features=expansion_factor * embed_size),
            nn.ReLU(),
            nn.Linear(in_features=expansion_factor * embed_size, out_features=embed_size)
        )
        self.dropout_2 = nn.Dropout(p=dropout)

        # LayerNorm 2
        self.layer_norm_2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = self.layer_norm_1(x)
        x = x + self.dropout_1(self.mha(x, x, x)) # Sub-Layer 1: Multi-Head Attention
        x = x + self.dropout_2(self.ffnn(self.layer_norm_2(x))) # Sub-Layer 2: Feed-Forward Neural Network
        return x

In [10]:
enc_block = EncoderBlock(num_heads=8, embed_size=embeddings_dim)
encoder_block_output = enc_block(final_word_embeddings)

encoder_block_output

tensor([[[ 1.4501,  1.4268, -1.1310,  ..., -0.0495, -1.4053, -1.6526],
         [ 0.0738,  0.7901, -1.9499,  ...,  2.2155, -1.2843,  1.8022],
         [ 1.2202,  1.0198, -1.0138,  ...,  1.1164, -1.1079, -0.1365],
         [ 0.7231, -1.4122,  0.2075,  ...,  0.1738, -0.6654,  1.8079],
         [-1.4398, -1.1762, -1.5248,  ...,  0.0305, -0.5861,  0.4337]]],
       grad_fn=<AddBackward0>)

In [11]:
encoder_block_output.shape

torch.Size([1, 5, 512])

### Decoder Block

#### Look-Ahead Mask

In [12]:
def look_ahead_mask(size: int):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask == 1

size = final_word_embeddings.shape[1]
mask = look_ahead_mask(size)
mask

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [13]:
embeddings_dim = 512

masked_mha = MultiHeadAttention(num_heads=8, embed_size=embeddings_dim)

masked_mha_output = masked_mha(final_word_embeddings, final_word_embeddings, final_word_embeddings, mask)
masked_mha_output # Each row in the output is the vector representation of a word

tensor([[[-0.1465, -0.5210,  0.4075,  ...,  0.0435,  0.9245,  0.0428],
         [-0.1359, -0.2841,  0.2091,  ...,  0.1719,  0.6713, -0.1088],
         [ 0.0199, -0.1481,  0.0224,  ...,  0.0733,  0.5482, -0.0702],
         [ 0.0402,  0.0698,  0.0727,  ...,  0.0360,  0.3538,  0.1240],
         [ 0.0130,  0.1138,  0.1161,  ..., -0.0299,  0.3039,  0.0737]]],
       grad_fn=<ViewBackward0>)

#### Prepare Decoder Input

In [14]:
def prepare_decoder_inputs(target_sequences, bos_token_id, eos_token_id):
    """
    Prepare shifted-right inputs for teacher forcing
    """
    # Remove <eos> from end, add <bos> to beginning
    decoder_input = torch.cat([
        torch.full((target_sequences.shape[0], 1), bos_token_id),  # Add <bos>
        target_sequences[:, :-1]  # Remove last token (<eos>)
    ], dim=1)
    return decoder_input

In [15]:
target_sequences = input_ids
decoder_input = prepare_decoder_inputs(target_sequences, bos_token_id=1, eos_token_id=2)
decoder_input

tensor([[1, 2, 3, 1, 5]])

#### Decoder Block

In [16]:
class DecoderBlock(nn.Module):
    def __init__(self, num_heads, embed_size, dropout=0.1, expansion_factor=4):
        super().__init__()
        self.masked_mha = MultiHeadAttention(num_heads=num_heads, embed_size=embed_size)
        self.cross_mha = MultiHeadAttention(num_heads=num_heads, embed_size=embed_size)

        self.layer_norm_1 = nn.LayerNorm(embed_size)
        self.layer_norm_2 = nn.LayerNorm(embed_size)
        self.layer_norm_3 = nn.LayerNorm(embed_size)

        self.ffnn = nn.Sequential(
            nn.Linear(in_features=embed_size, out_features=expansion_factor * embed_size),
            nn.ReLU(),
            nn.Linear(in_features=expansion_factor * embed_size, out_features=embed_size)
        )

        self.dropout_1 = nn.Dropout(p=dropout)
        self.dropout_2 = nn.Dropout(p=dropout)
        self.dropout_3 = nn.Dropout(p=dropout)

    def forward(self, x, mask, encoder_output):
        x = self.layer_norm_1(x)
        x = x + self.dropout_1(self.masked_mha(x, x, x, mask=mask))
        x = self.layer_norm_2(x)
        x = x + self.dropout_2(self.cross_mha(x, encoder_output, encoder_output))
        x = x + self.dropout_3(self.ffnn(self.layer_norm_3(x)))
        return x

In [17]:
dec_block = DecoderBlock(num_heads=8, embed_size=embeddings_dim)
decoder_block_output = dec_block(final_word_embeddings, mask, encoder_block_output)

decoder_block_output

tensor([[[ 1.7671,  1.0041, -0.9346,  ..., -0.2916, -0.8281, -1.4732],
         [ 0.0824,  1.0855, -2.1151,  ...,  2.2532, -1.3636,  1.6164],
         [ 1.4923,  1.1168, -0.6429,  ...,  0.6819, -0.6023, -0.7787],
         [ 0.9411, -1.3277,  0.9624,  ...,  0.5686, -1.0313,  1.5216],
         [-0.6372, -1.1567, -0.6685,  ...,  0.3843, -0.7720,  0.2854]]],
       grad_fn=<AddBackward0>)

## Transformer

In [18]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, num_heads, embed_size, num_layers=1):
        super().__init__()
        self.enc_layer = nn.ModuleList([
            EncoderBlock(num_heads=num_heads, embed_size=embed_size, dropout=0.1, expansion_factor=4)
            for _ in range(num_layers)
        ])
        self.dec_layer = nn.ModuleList([
            DecoderBlock(num_heads=num_heads, embed_size=embed_size, dropout=0.1, expansion_factor=4)
            for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(in_features=embed_size, out_features=tgt_vocab_size)
        self.encoder_embeddings = nn.Embedding(embedding_dim=embed_size, num_embeddings=src_vocab_size)
        self.decoder_embeddings = nn.Embedding(embedding_dim=embed_size, num_embeddings=tgt_vocab_size)
        self.embed_size = embed_size

    def positional_encoding(self, seq_length, d, n=10_000):
        P = torch.zeros((seq_length, d)) # Initiate the matrix

        pos = torch.arange(seq_length).unsqueeze(1) # Get all positions
        dim_idx = torch.arange(0, d, 2) # Get all dimensions

        denominator = torch.pow(n, dim_idx/d) # Calculate the denominator (n^(2i/d))
        P[:, 0::2] = torch.sin(pos/denominator) # Get the positional encoding for all even positions
        P[:, 1::2] = torch.cos(pos/denominator) # Get the positional encoding for all odd positions
        return P

    def forward(self, encoder_input, decoder_input, mask):
        batch_size, src_seq_len = encoder_input.shape # Shape of encoder's input
        _, tgt_seq_len = decoder_input.shape # Shape of decoder's input

        # Encoder Block
        src_P = self.positional_encoding(seq_length=src_seq_len, d=self.embed_size) # Positional Encoding
        encoder_embeddings = self.encoder_embeddings(encoder_input) # Trainable Word Embeddings
        encoder_full_embeddings = encoder_embeddings + src_P # Final Word Embeddings (Word Embeddings + Positional Encoding)
        encoder_output = encoder_full_embeddings
        for layer in self.enc_layer:  # Fixed variable name
            encoder_output = layer(encoder_output)

        # Decoder Block
        trgt_P = self.positional_encoding(seq_length=tgt_seq_len, d=self.embed_size) # Positional Encoding
        decoder_embeddings = self.decoder_embeddings(decoder_input) # Trainable Word Embeddings
        decoder_full_embeddings = decoder_embeddings + trgt_P # Final Word Embeddings (Word Embeddings + Positional Encoding)
        decoder_output = decoder_full_embeddings
        for layer in self.dec_layer:  # Fixed variable name
            decoder_output = layer(decoder_output, mask, encoder_output)

        # Output
        fc_output = self.fc_out(decoder_output)
        return fc_output

In [19]:
transfomer = Transformer(src_vocab_size=10, tgt_vocab_size=10, num_heads=8, embed_size=embeddings_dim)
transfomer_output = transfomer(encoder_input=input_ids, decoder_input=decoder_input, mask=mask)
transfomer_output

tensor([[[ 0.8935,  0.6845,  0.1212, -0.8010, -0.3575,  0.4422, -0.1093,
           0.2551,  0.4357,  0.3551],
         [-0.5662, -0.0265,  0.7028, -0.0780,  0.0679,  0.2428, -0.7713,
          -1.3900,  0.2802, -0.3520],
         [ 1.5709,  0.7458,  1.3701,  0.5477, -0.3777,  0.2054,  0.0711,
          -0.7186, -0.6136, -0.8580],
         [ 1.2695,  0.6599,  0.1023, -0.1835, -0.4030,  0.7427, -0.0130,
           0.6603,  0.1303,  0.5052],
         [-0.5066,  0.8474,  0.1716, -0.0092,  0.1736, -0.5638,  0.0044,
          -1.4808, -0.5354, -0.3485]]], grad_fn=<ViewBackward0>)